# The goose Agent in VS Code

In this course we use [goose](https://goose-docs.ai), an open-source AI agent, inside VS Code. goose passes our requests to a large-language model (LLM) running on College infrastructure (the HX3 cluster), as configured in [00-setup.ipynb](00-setup.ipynb). We do not use commercial tools such as [GitHub Copilot](https://docs.github.com/en/copilot) or [Microsoft Copilot](https://copilot.microsoft.com/); if VS Code offers to install Copilot or sign in to GitHub, you can say no.

Using our own model has some practical consequences:
* There is no subscription: access is through your HX3 account and a token (API key) that expires after about an hour.
* Your code and prompts go to the College cluster, not to a third-party company.
* The model served on HX3 (e.g. Qwen) is smaller than the largest commercial models, so check its output carefully.

goose is an *agent*: as well as answering questions, it can read files, create and edit files and run commands, asking your permission first. Because it integrates into your editor, it is much more powerful and easier to use for programming than a standalone chat window.

In this course, we will be using it in VS Code in relation to the Python programming language. However, the model supports many programming languages.

In this notebook, we'll look at the goose interface. In future notebooks, we'll look at how to use it to achieve specific tasks when programming.

## Implementation in VS Code

VS Code has many extensions that can be installed to add additional functionality. These can be accessed via the marketplace, which is accessible from the extensions tab on the left-hand side of the screen.

goose is used through the **VS Code Goose extension** (publisher: Block), which talks to the goose CLI installed on your computer. Both were installed and configured in [00-setup.ipynb](00-setup.ipynb). The extension is still experimental, so its interface may change slightly between versions.

Note that goose does not provide code autocompletion as you type: you interact with it through its chat panel.

## Generating Code

Instead of completing code as you type, you describe what you want in the goose chat and it writes the code for you. You can then copy the code into your file using the copy button on the code block.

**Task**: Open the goose chat (see *Chat Interface* below), ask it to write a Python function that converts a temperature from Celsius to Kelvin, and copy the result into the code cell below:

## Finding Errors

goose can also look at existing code and point out problems. To send code to goose, highlight it in the editor and press ctrl + shift + G (cmd + shift + G on Mac). The selected code appears as a context "chip" above the chat input.

**Task**: The function in the cell below contains an error, where the operation of the function is at odds with the name. Highlight the function, send it to goose, and ask it whether the code is correct.

In [ ]:
def add(x, y):
    return x - y

## Chat Interface

The goose chat interface appears in the side panel. If it's not open, click the goose icon in the activity bar on the left of the editor. This will open a chat interface where you can ask questions about your code, or ask the agent to perform different tasks. You can drag the panel to the right-hand side of the editor if you prefer.

### Context

You can give goose context in two ways: highlight code in a file and press ctrl/cmd + shift + G to send the selection, or type `@` in the chat box followed by the start of a file name to attach a whole file. Each item appears as a chip above the chat input, and it is possible to add several in this way. By providing appropriate context, you can get more relevant responses more quickly and with a lower token usage. Briefly explaining the relevance of each piece of context can also help explain the situation to the model and improve its responses.

**Task**: Try adding this whole notebook as context and asking goose to give you a high-level overview of the file.

In [ ]:
def find_unique_characters(string):
    characters = set()
    for char in string:
        characters.add(char)

    characters = list(characters)
    characters.sort()

    return characters

print(find_unique_characters("hello"))
print(find_unique_characters("Imperial College London"))

### Using Generated Code

If you ask goose to add or edit code in a file, it will change the file itself, rather than showing a suggestion for you to accept. Because we configured goose in **Approve Mode--- CORRECT?**, it will ask for your permission before it modifies a file or runs a command, and you can allow or deny each action. You should review these changes for correctness. After a change has been made, you can see exactly what changed (and undo it) in VS Code's *Source Control* view if the folder is a git repository.

Notebook (`.ipynb`) files are stored in a complex format that an agent can easily damage, so it is safer to let goose edit plain Python (`.py`) files, or to copy its code into the notebook yourself.

**Task**: Add the code below as context, and ask goose to improve it. Copy the improved version into a new code cell and compare the two.

In [ ]:
def triple(x):
    return x + x + x

### Modes

goose has several modes that control how much it is allowed to do without asking you:
* **Auto**: goose can create, edit and delete files and run commands without asking. This is risky, and we do not use it in this course.
* **Approve**: goose asks for your approval before every file modification, command or tool use. This is the mode we chose during setup.
* **Smart Approve**: goose decides which actions are low risk (e.g. reading a file) and only asks for approval for the others.
* **Chat**: goose only talks; it does not use tools or modify files.

The mode can be changed by running `goose configure` in a terminal and choosing *goose settings* → *goose mode*. In a terminal session (`goose session`) you can also switch with `/mode approve`, `/mode chat`, etc., and use `/plan` to ask goose to propose a plan before it makes any changes.

It is also possible to give goose extra capabilities through extensions and to define reusable instructions ("recipes"), but that is beyond the scope of this introduction.

### Models

goose uses the model that we selected during setup, which is served by HX3 (the provider is set to *OpenAI* because HX3 offers an OpenAI-compatible interface). The model is recorded as `GOOSE_MODEL` in goose's `config.yaml` file, and can be changed by running `goose configure` again if other models are made available on HX3.

Larger models generally give better answers, especially for complex reasoning and for agent tasks that require using tools, but respond more slowly. Very small models may struggle to use goose's tools correctly, so if goose describes an action instead of carrying it out, the model is a likely cause.

**Task**: Open goose's `config.yaml` (in `%APPDATA%\Block\goose\config\` on Windows) and find which provider and model you are using.

### Context Window

An agent's working memory is defined by its context window. This is its memory of the current conversation, the files it has looked at, any context and instructions. If the context window fills, goose summarises the earlier conversation, effectively compressing the information it has seen into a smaller representation. This summary will be used in place of the original context, which can lead to the agent forgetting details.

When you prompt the agent, it will examine the contents of its context window and use that information to generate a response. As the window fills, this will increase the token usage of each prompt and the latency of the response.

The size of the context window is set by how the model is deployed on HX3, and is usually smaller than for commercial services, so it is worth keeping the context focused.

### Starting a New Session

It's often a good idea to start a new session when changing tasks due to the behaviour of the context window. This can minimise token usage and improve response times. However, this will also clear the context window, so you will need to provide any relevant context again. You can do this by starting a new session from the goose panel.

**Task**: Try opening a new session now.

### Revisiting Old Sessions

goose keeps a history of your sessions, which you can open from the goose panel. You can click on a session to open it and continue the conversation. This means you can have several sessions in use and, if you want to revisit a task you have previously worked on, you can open the relevant session and continue the conversation with its history intact.

**Task**: Try switching back to your original session.

### Tokens and Usage

There are no usage credits: the model runs on HX3. However, the token that gives goose access to HX3 expires after about one hour. If goose suddenly reports authentication errors, generate a new token as described in [00-setup.ipynb](00-setup.ipynb) (or check that the *HX3_tokens.py* script is configured). The cluster is a shared resource, so responses may be slower when many people are using it.

## Privacy and Security

It's important to be aware of the privacy and security implications of any Generative AI tool. Because the model runs on College infrastructure, your code and prompts are not sent to a commercial provider, which makes this set-up suitable for unpublished research code. They do, however, leave your computer and are processed on the HX3 cluster.

Some precautions still apply:
* Your Client_ID/Client_secret pair and your token are equivalent to a password. Keep *HX3_tokens.py* and goose's configuration private, and never paste them into a chat.
* goose can read and modify files and run commands on your computer. Keep it in Approve Mode, read each request before allowing it, and open goose only in the folders you want it to work on.
* During setup we chose not to share anonymous usage data with the goose developers.

Imperial provides [more information on protecting sensitive data in general](https://www.imperial.ac.uk/research-and-innovation/support-for-staff/scholarly-communication/research-data-management/data-storage-and-security/storing-sensitive-and-personal-data/).

## Reducing Plagiarism

Open-weight models such as Qwen were trained on large amounts of public code, including code under licenses that restrict duplication and/or modification. They can occasionally reproduce such code closely. Unlike some commercial services, our set-up has no filter that blocks suggestions matching public code, so the responsibility for checking lies with you. The fact that models can do this is an ethical issue with potential legal implications that have not been fully explored yet.

Even when not copied exactly, suggested code will still be inspired by the code in public repositories, including code under a license which prevents duplication and/or modification. This can still be considered an ethical issue, but most would consider it a lesser issue than outright duplication of the code.

The ethics and legality of generative AI of all sorts are a matter of debate in the context of plagiarism. As a result, there is not a "correct" stance to communicate to you, but it is an issue you should be aware of and consider.

## Exercise

Take some time to get used to the goose interface. You might try to:

* Ask goose to write a function from a short description. You may choose a function relevant to your research project.
* Ask goose a general question about programming.
* Ask goose a question about a specific piece of code.
* Add some new code generated by asking a question in the chat window to the code cell below.